## Filtering

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Read image
img = cv2.imread('Chapter1/images/image.png')

# Convert BGR to RGB for display
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Function to show the image
def show(img, title="Image"):
    plt.figure(figsize=(5,5))
    if len(img.shape) == 2:  
        # grayscale image
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(img)
    plt.title(title)
    plt.axis("off")
    plt.show()

1. Mean Filter (Average Filter)

The mean filter replaces each pixel with the average value of its neighboring pixels.

In [ ]:
# Mean Filter using a 5x5 kernel
mean_filtered = cv2.blur(img_rgb, (5, 5))

show(img_rgb, "Original Image")
show(mean_filtered, "Mean Filtered Image")

Alternative using convolution kernel:

In [ ]:
kernel = np.ones((5, 5), np.float32) / 25

mean_filtered = cv2.filter2D(img_rgb, -1, kernel)

show(mean_filtered, "Mean Filtered Image")

2. Weighted Filter

A weighted filter assigns different weights to neighboring pixels.

In [ ]:
kernel = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1]
], dtype=np.float32)

kernel = kernel / kernel.sum()

weighted_filtered = cv2.filter2D(img_rgb, -1, kernel)

show(weighted_filtered, "Weighted Filtered Image")

3. Gaussian Filter

Gaussian filtering smooths an image using a Gaussian distribution.

In [ ]:
gaussian_filtered = cv2.GaussianBlur(
    img_rgb,
    (5, 5),   # kernel size
    sigmaX=1.0
)

show(gaussian_filtered, "Gaussian Filtered Image")

Non Linear Filters

4. Maximum Filter

The maximum filter replaces each pixel with the maximum value in its neighborhood. It is implemented using dilation.

In [ ]:
kernel = np.ones((5, 5), np.uint8)

maximum_filtered = cv2.dilate(img_rgb, kernel)

show(maximum_filtered, "Maximum Filtered Image")

5. Minimum Filter

The minimum filter replaces each pixel with the minimum value in its neighborhood. It is implemented using erosion.

In [ ]:
kernel = np.ones((5, 5), np.uint8)

minimum_filtered = cv2.erode(img_rgb, kernel)

show(minimum_filtered, "Minimum Filtered Image")

6. Median Filter

The median filter replaces each pixel with the median value of neighboring pixels. It is very effective for removing salt-and-pepper noise.

In [ ]:
median_filtered = cv2.medianBlur(
    img_rgb,
    5   # kernel size (must be odd)
)

show(median_filtered, "Median Filtered Image")

In [ ]:
filters = [
    ("Original", img_rgb),
    ("Mean", cv2.blur(img_rgb, (5, 5))),
    ("Weighted", cv2.filter2D(
        img_rgb,
        -1,
        np.array([[1,2,1],[2,4,2],[1,2,1]], np.float32)/16
    )),
    ("Gaussian", cv2.GaussianBlur(img_rgb, (5,5), 1)),
    ("Maximum", cv2.dilate(img_rgb, np.ones((5,5), np.uint8))),
    ("Minimum", cv2.erode(img_rgb, np.ones((5,5), np.uint8))),
    ("Median", cv2.medianBlur(img_rgb, 5))
]

plt.figure(figsize=(14, 8))

for i, (title, image) in enumerate(filters, start=1):
    plt.subplot(2, 4, i)
    plt.imshow(image)
    plt.title(title)
    plt.axis('off')

plt.tight_layout()
plt.show()

### Sharpening

#### Unsharp masking. 

The idea is:
Smooth the original image.

Extract details:
Details=Original−Smoothed

Add details back to the original:
Sharpened=Original+Details

or more generally,

Sharpened=Original+k×Details

where k controls the sharpening strength.

1. Sharpening Using Mean Filter

In [ ]:
# Step 1: Smooth image
smoothed = cv2.blur(img_rgb, (5, 5))

# Step 2: Extract details
details = cv2.subtract(img_rgb, smoothed)

# Step 3: Add details back
sharpened = cv2.add(img_rgb, details)

show(smoothed, "Mean Smoothed")
show(details, "Details")
show(sharpened, "Sharpened (Mean)")

2. Sharpening Using Weighted Filter

In [ ]:
kernel = np.array([
    [1,2,1],
    [2,4,2],
    [1,2,1]
], dtype=np.float32)

kernel = kernel / kernel.sum()

smoothed = cv2.filter2D(img_rgb, -1, kernel)

details = cv2.subtract(img_rgb, smoothed)

sharpened = cv2.add(img_rgb, details)

show(smoothed, "Weighted Smoothed")
show(details, "Details")
show(sharpened, "Sharpened (Weighted)")

3. Sharpening Using Gaussian Filter

In [ ]:
smoothed = cv2.GaussianBlur(img_rgb, (5, 5), 1)

details = cv2.subtract(img_rgb, smoothed)

sharpened = cv2.add(img_rgb, details)

show(smoothed, "Gaussian Smoothed")
show(details, "Details")
show(sharpened, "Sharpened (Gaussian)")

Adjustable Sharpening Strength

Instead of adding the full detail image, use a scaling factor k.

In [ ]:
smoothed = cv2.GaussianBlur(img_rgb, (5,5), 1)

details = cv2.subtract(img_rgb, smoothed)

k = 2.0  # sharpening strength

sharpened = cv2.addWeighted(
    img_rgb, 1.0,
    details, k,
    0
)

show(sharpened, "Sharpened (k = 2.0)")